In [ ]:
import random

def generar_datos_legacy(nombre_archivo="envios_legacy.txt", cantidad_registros=500000):
    """
    Genera un archivo de texto simulando un volcado de base de datos antiguo.
    Incluye ruido y errores intencionales para probar pipelines de datos.
    """
    estados = ["ENTREGADO", "EN_RUTA", "CANCELADO", "entregado", " cancelado ", " EN_RUTA"]
    zonas = ["Norte", "Sur", "Este", "Oeste", "Centro", "norte", " SUR "]

    print(f"[*] Generando {cantidad_registros} registros de prueba...")

    with open(nombre_archivo, "w", encoding="utf-8") as f:
        f.write("ID_PAQUETE| ESTADO | DISTANCIA_KM | COSTO_USD | ZONA\n")
        
        for i in range(1, cantidad_registros + 1):
            estado = random.choice(estados)
            distancia = round(random.uniform(2.0, 150.0), 1)
            costo = random.randint(5, 800)
            zona = random.choice(zonas)
            
           
            if i % 1000 == 0:
                f.write(f"PKG{i}|FATAL_ERROR_SISTEMA_CAIDO_LINEA_INCOMPLETA\n")

            elif i % 1500 == 0:
                f.write(f" PKG{i} | {estado} | {distancia} | NULL_VALUE | {zona} \n")

            else:
                f.write(f" PKG{i} | {estado} |  {distancia} | {costo} | {zona} \n")

    print(f"[+] ¡Éxito! Archivo '{nombre_archivo}' creado. Ya podés correr tu ETL.")

generar_datos_legacy()

[*] Generando 500000 registros de prueba...
[+] ¡Éxito! Archivo 'envios_legacy.txt' creado. Ya podés correr tu ETL.


In [ ]:
import requests # librería para hacer solicitudes HTTP
import pandas as pd # librería para manipulación de datos
from IPython.display import display # función para mostrar dataframes de forma más legible

def reintentar_api(func): # decorador para reintentar la función en caso de error
    def wrapper(*args, **kwargs): # función envoltorio que maneja los reintentos
        for i in range(3): # intentamos ejecutar la función hasta 3 veces
            try: # si la función se ejecuta sin errores, devolvemos su resultado
                return func(*args, **kwargs) # si ocurre un error, lo capturamos y mostramos un mensaje de reintento
            except Exception: # si ocurre cualquier excepción, mostramos un mensaje de error y reintentamos
                print("error, reintentando") # si después de 3 intentos sigue fallando, devolvemos None
        return None # devolvemos None si la función no se ejecutó correctamente después de 3 intentos
    return wrapper # devolvemos la función envoltorio para que pueda ser usada como decorador

@reintentar_api  # aplicamos el decorador a la función que obtiene el valor del dólar
def obtener_dolar(): # función para obtener el valor de venta del dólar oficial desde una API
    r = requests.get("https://dolarapi.com/v1/dolares/oficial") # hacemos una solicitud GET a la API para obtener los datos del dólar oficial
    data = r.json() # convertimos la respuesta de la API a formato JSON para poder acceder a sus datos
    return data["venta"] # devolvemos el valor de venta del dólar oficial obtenido de la API

def leer_archivo(ruta): # función para leer el archivo de texto y generar un diccionario por cada línea con los datos relevantes
    with open(ruta, "r") as f: # abrimos el archivo en modo lectura
        next(f) # saltamos la primera línea (encabezado)

        for linea in f: # iteramos sobre cada línea del archivo para procesarla
            try: # intentamos procesar la línea, si ocurre un error, lo capturamos y continuamos con la siguiente línea
                partes = linea.split("|") # dividimos la línea en partes usando el carácter "|" como separador

                if len(partes) != 5: # si la línea no tiene exactamente 5 partes, es un formato incorrecto, así que la ignoramos y pasamos a la siguiente línea
                    continue # verificamos que la línea tenga el formato correcto antes de procesarla

                id = partes[0].strip() # obtenemos el ID del paquete, eliminando espacios en blanco al inicio y al final
                estado = partes[1].strip().upper() # obtenemos el estado del paquete y lo convertimos a mayúsculas
                distancia = float(partes[2].strip()) # obtenemos la distancia y la convertimos a float
                costo = partes[3].strip() # obtenemos el costo del paquete
                zona = partes[4].strip().upper() # obtenemos la zona y la convertimos a mayúsculas

                if costo == "NULL_VALUE": # si el costo es NULL_VALUE, lo ignoramos
                    continue # si el costo no es un valor numérico válido, lo ignoramos

                costo = float(costo) # convertimos el costo a float para poder realizar cálculos posteriores

                yield { # devolvemos un diccionario con los datos procesados de la línea
                    "id": id, # el ID del paquete
                    "estado": estado, # el estado del paquete (ENTREGADO, EN_RUTA, CANCELADO)
                    "distancia": distancia, # la distancia en kilómetros
                    "costo": costo, # el costo en dólares
                    "zona": zona # la zona de entrega (NORTE, SUR, ESTE, OESTE, CENTRO)
                }

            except Exception: # si ocurre cualquier error al procesar la línea, lo capturamos y mostramos un mensaje de error, luego continuamos con la siguiente línea
                continue # si ocurre un error al procesar la línea, simplemente la ignoramos y seguimos con la siguiente línea

def transformar(datos): # función para transformar los datos obtenidos del archivo

    dolar = obtener_dolar() # obtenemos el valor del dólar oficial

    filtrados = filter( # filtramos los datos para quedarnos solo con los paquetes que no están cancelados y tienen una distancia mayor o igual a 20 km
        lambda x: x["estado"] != "CANCELADO" and x["distancia"] >= 20, # aplicamos un filtro para quedarnos solo con los paquetes que no están cancelados y tienen una distancia mayor o igual a 20 km
        datos # aplicamos el filtro a los datos obtenidos del archivo para generar un nuevo iterable con solo los registros que cumplen las condiciones establecidas
    )

    transformados = map( # transformamos los datos filtrados para calcular el costo final en pesos argentinos, aplicando el valor del dólar y el impuesto del 21%
        lambda x: {
            "id": x["id"], # el ID del paquete
            "zona": x["zona"], # la zona de entrega (NORTE, SUR, ESTE, OESTE, CENTRO)
            "costo_final": x["costo"] * dolar * 1.21 # calculamos el costo final en pesos argentinos, aplicando el valor del dólar y el impuesto del 21%

        },
        filtrados # aplicamos la transformación a los datos filtrados para generar un nuevo iterable con los registros transformados, incluyendo el cálculo del costo final en pesos argentinos
    )

    ordenados = sorted( # ordenamos los datos transformados por costo final de forma descendente para obtener un ranking de los paquetes más costosos
        transformados, # aplicamos la función sorted para ordenar los datos transformados por costo final de forma descendente, generando una lista ordenada con los registros transformados
        key=lambda x: x["costo_final"], # especificamos que la clave de ordenamiento es el costo final para ordenar los registros por ese campo
        reverse=True # indicamos que el ordenamiento debe ser descendente para obtener un ranking de los paquetes más costosos, colocando los registros con mayor costo final al inicio de la lista ordenada
    )

    return ordenados # devolvemos la lista ordenada con los registros transformados, incluyendo el ID del paquete, la zona de entrega y el costo final en pesos argentinos, ordenados por costo final de forma descendente

def guardar(datos, ruta): # función para guardar los datos transformados en un archivo CSV, mostrando un dataframe con los datos antes de guardarlos
    df = pd.DataFrame(datos) # creamos un dataframe de pandas a partir de los datos transformados para facilitar su manipulación y visualización antes de guardarlos en un archivo CSV

    df["costo_final"] = df["costo_final"].round(2) # redondeamos el costo final a 2 decimales para mejorar su presentación en el dataframe y en el archivo CSV, facilitando su lectura y análisis posterior

    display(df) # mostramos el dataframe con los datos transformados antes de guardarlos en un archivo CSV para verificar su contenido y formato, permitiendo una revisión visual de los registros procesados antes de su almacenamiento definitivo

    df.to_csv(ruta, index=False, sep=";") # guardamos el dataframe en un archivo CSV en la ruta especificada, sin incluir el índice y utilizando el punto y coma como separador

def main(): # función principal que ejecuta el proceso completo de lectura, transformación y guardado de los datos, mostrando un mensaje al finalizar el proceso
    datos = leer_archivo("envios_legacy.txt") # leemos los datos del archivo de envíos legacy
    procesados = transformar(datos) # transformamos los datos leídos para calcular el costo final en pesos argentinos
    guardar(procesados, "reporte_logistica_limpio.csv") # guardamos los datos transformados en un archivo CSV
    print("Proceso terminado") # mostramos un mensaje indicando que el proceso ha finalizado
    
main() # ejecutamos la función principal para iniciar el proceso completo de lectura, transformación y guardado de los datos, mostrando un mensaje al finalizar el proceso

,id,zona,costo_final
0,PKG2012,SUR,1374560.0
1,PKG3452,OESTE,1374560.0
2,PKG5604,SUR,1374560.0
3,PKG5634,SUR,1374560.0
4,PKG10925,NORTE,1374560.0
...,...,...,...
292282,PKG494105,ESTE,8591.0
292283,PKG495095,CENTRO,8591.0
292284,PKG498719,NORTE,8591.0
292285,PKG498966,SUR,8591.0


Proceso terminado
